# Visium HD Artifact Cleanup

Demonstration notebook showing how to load a Visium HD sample with `Artifact_remove_HD`,
remove artifacts, and save the updated outputs.


In [1]:
import os, sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from artifactsRemoval import Artifact_remove_HD

data_dir = '/path/to/Visium_HD/sample'  # <- update this path
data_dir ="/users/2/wan00232/data_folder/Visium_HD/binned_outputs/square_002um"
positions_path = os.path.join(data_dir, 'spatial', 'tissue_positions.parquet')
h5_path = os.path.join(data_dir, 'filtered_feature_bc_matrix.h5')
print('Using data_dir:', data_dir)
print('Positions exists?', os.path.exists(positions_path))
print('HDF5 exists?', os.path.exists(h5_path))


Using data_dir: /users/2/wan00232/data_folder/Visium_HD/binned_outputs/square_002um
Positions exists? True
HDF5 exists? True


In [2]:
import time 


In [3]:
start_time = time.perf_counter()
hd = Artifact_remove_HD(
    data_dir=data_dir,
    positions_path=positions_path,
    h5_path=h5_path,
    neighbor_mode='k',
    k_neighbors=8,
)

print('Barcodes:', hd.barcode_list.shape[0])
print('Features:', hd.feature_list.shape[0])
print('Positions:', hd.tissue_position.shape[0])
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Function executed in: {elapsed_time:.6f} seconds")


Barcodes: 6296688
Features: 19070
Positions: 11222500
Function executed in: 805.311923 seconds


## Outlier Detector Check

Measure the adaptive outlier detector to confirm it converges quickly on the HD grid.


In [4]:
import warnings

summary_df = hd.get_sum()
start = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', RuntimeWarning)
    outliers = Artifact_remove_HD.outlier(summary_df)
duration = time.perf_counter() - start
rate = len(outliers) / len(summary_df) if len(summary_df) else 0.0

print(f'Detected {len(outliers)} outlier spots ({rate:.2%}) in {duration:.2f} seconds')
if caught:
    for w in caught:
        print(f'Warning: {w.message}')


Detected 161829 outlier spots (2.57%) in 58.40 seconds


In [5]:
# Remove artifacts
start_time = time.perf_counter()
hd.remove_border(distance=float(hd._grid_radius) if hd._grid_radius else 0.0)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Function executed in: {elapsed_time:.6f} seconds")


start_time = time.perf_counter()
hd.remove_edge(distance=2)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Function executed in: {elapsed_time:.6f} seconds")


start_time = time.perf_counter()
hd.remove_malfunction()  # clustered removal (min cluster size 15)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Function executed in: {elapsed_time:.6f} seconds")


We removed 237 border spots within border width 8.02827171223159
Function executed in: 313.336454 seconds
We removed 32600 edge spots within 2 layer(s)
Function executed in: 246.975984 seconds


/users/2/wan00232/ondemand/jupyter/Visium/QC/Gitversion/BLADE/artifactsRemoval/Artifact_remove_HD.py:728: RuntimeWarning: Outlier detection stopped after 200 iterations (target rate=0.050, observed=0.026). Results may deviate from the target rate.
  warnings.warn(


Clustered malfunction removal: considered 159721 outliers, removed 2430 spots in clusters >= 15 (radius=8.028).
Function executed in: 380.173653 seconds


In [7]:
# Save outputs
output_dir = os.path.join(data_dir, 'blade_outputs')
hd.save(output_dir)
out_h5 = os.path.join(output_dir, 'filtered_feature_cell_matrix.filtered.h5')
# Save filtered matrix (keeps only in-tissue spots by default)
hd.save_hdf5(out_h5, filter_in_tissue=True)


Saved the tissue position file to /users/2/wan00232/data_folder/Visium_HD/binned_outputs/square_002um/blade_outputs/tissue_positions.csv
Saved Visium HD HDF5 to /users/2/wan00232/data_folder/Visium_HD/binned_outputs/square_002um/blade_outputs/filtered_feature_cell_matrix.filtered.h5 (retained 6261421 of 6296688 spots).
